# Reentrenamiento v4 — RoBERTuito + datos reales de X (octubre 2026)

**Pregunta:** ¿agregar las 4,074 filas reales de X mejora al sistema de la Etapa II, o solo lo cambia?

**Diseño:** dos corridas con el **mismo código, el mismo archivo de train y las mismas semillas** (42, 1337, 2026). La única diferencia es `--peso_nuevos`:

| Corrida | `--peso_nuevos` | Train efectivo |
|---|---|---|
| `base` | 0.0 | 6,395 = train de la Etapa II (6,416) − 21 filas que solapaban con val/batería |
| `v4` | 1.0 | 10,469 = lo anterior + 4,074 de octubre |

Receta = la de la Etapa II (`src/v3/entrenar.py`, repo v3): `robertuito-base-cased`, texto `texto_modelo` sin pasar a minúsculas, lr 5.73e-5, batch 32, 6 épocas, early stopping 2, pesos por clase, sintéticos con peso 0.75, ensemble = promedio de las 3 semillas.

**Antes de correr:** `Entorno de ejecución → Cambiar tipo → GPU`. Luego `Ejecutar todo`. Si Colab se desconecta, volvé a ejecutar todo: con Drive montado, las semillas ya terminadas se reutilizan.

**Tiempo:** no medido en Colab. En la Etapa II cada semilla tardó 1.7–2.5 min con 6,416 filas en otra GPU; aquí son 6 entrenamientos, uno de ellos con 1.6 veces más filas.

In [ ]:
# 0. GPU y carpeta de salida (Drive, para sobrevivir a una desconexión)
USAR_DRIVE = True

import subprocess, os
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout or "SIN GPU")
import torch
assert torch.cuda.is_available(), "Activá la GPU: Entorno de ejecución → Cambiar tipo de entorno → GPU"

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/toxicidad_v4/runs"
else:
    RUNS = "/content/runs"
os.makedirs(RUNS, exist_ok=True)
print("salidas en", RUNS)

In [ ]:
# 1. Dependencias fijadas. emoji y ftfy en estas versiones reproducen byte a byte
#    el texto_modelo guardado del train y del test de la Etapa II (verificado).
!pip install -q "transformers==4.46.3" "accelerate==1.2.1" "emoji==2.16.0" "ftfy==6.3.1"
import transformers, emoji, sklearn, pandas
print("torch", torch.__version__, "· transformers", transformers.__version__, "· emoji", emoji.__version__,
      "· sklearn", sklearn.__version__, "· pandas", pandas.__version__)

In [ ]:
# 2. Repo del equipo, rama v3, en el commit exacto de la Etapa II
COMMIT = "95d3ba63f40413e493cc0df174ea5b9cc2da7f47"
%cd /content
!rm -rf proyecto && git clone -q https://github.com/caeher/cde-project-ml-2026 proyecto
%cd /content/proyecto
!git checkout -q {COMMIT} && git log -1 --format="%h %ad %s"

In [ ]:
# 3. Archivos nuevos embebidos: los 3 CSV de octubre y los 3 scripts corregidos.
#    Se verifica el SHA-256 de cada uno al extraerlos.
import base64, io, tarfile, hashlib
PAYLOAD = "..."
tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz").extractall(".")
ESPERADO = {
    "data/sources/x_oct2026/dataset_x2000.csv": "e984b31b9de264299f4c90c1757780fd1b83cbdb252db71c2a047b0a5729a4ae",
    "data/sources/x_oct2026/lenguaje_ofensivo.csv": "e267799fbb9551a11c81ee37a780cd71f15762fc6070a1cb5e2fa688e8efec28",
    "data/sources/x_oct2026/thenewdata_X.csv": "891d6dc62287351a2bb1afe5879129bf87f67bae2c24edc3c87f936ae86e9d8c",
    "src/data/build_dataset_v4.py": "7afbb6ae3dc75577c6f4339ff2f2c422ae86bd0e97f2baf4a88a084317a3d151",
    "src/train/entrenar_v4.py": "cebe33471de5a8f3d7bf962856443e6f38bb8827165518fa9d6dd8461a083f7c",
    "src/evaluation/comparar_v4.py": "120b9298453ff265a41a0785cfcfc4f1d705186aaff5db6cd66b815f5908cecc"
}
for ruta, h in ESPERADO.items():
    real = hashlib.sha256(open(ruta, "rb").read()).hexdigest()
    assert real == h, f"{ruta}: hash distinto"
    print("ok", ruta)

In [ ]:
# 4. Controles previos: test intacto, train de la Etapa II, normalizador v1.2
import hashlib, sys, pandas as pd
sys.path.insert(0, ".")
from src.features.normalization import normalizar
import src.features.normalization as N

sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
assert sha("data/history/v1.2.0/test.csv").startswith("5d1284efcc7df20f"), "el test no es el de julio"
assert sha("data/history/v1.2.0/val.csv").startswith("45aabe9744711d74"), "val no es el de julio"
assert sha("data/history/v1.2.0/train.csv").startswith("4e50ca740a0ef88a"), "train base distinto al de la Etapa II"
assert hasattr(N, "CORRECCIONES_EMOJI"), "normalizador viejo (v1.1): falta la corrección de 🔫 y arroba"

for f in ["train", "test"]:
    d = pd.read_csv(f"data/history/v1.2.0/{f}.csv")
    iguales = (d.texto_original.astype(str).map(normalizar) == d.texto_modelo.astype(str)).sum()
    print(f"{f}: normalizar(texto_original) == texto_modelo guardado en {iguales}/{len(d)}")
    assert iguales == len(d), "la normalización no reproduce la de la Etapa II: revisar versión de emoji"
print("controles ok · train base", len(pd.read_csv("data/history/v1.2.0/train.csv")), "filas")

In [ ]:
# 5. Dataset v4 sobre el train de 6,416 de la Etapa II
#    (el script aborta si algún texto de train coincide con val, test o la batería)
!python src/data/build_dataset_v4.py --base data/history/v1.2.0/train.csv --salida data/dataset_v4_e2 && python -c "import json; m=json.load(open('data/dataset_v4_e2/metadata.json')); assert m['train_v4_total']==10469"

In [ ]:
# 6. Verificación independiente de fugas y de la composición
import re, unicodedata
def clave(s):
    s = unicodedata.normalize("NFKC", str(s)).lower()
    s = re.sub(r"https?://\S+", " ", s); s = re.sub(r"@\w+", " ", s); s = re.sub(r"[^\w\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

D = "data/dataset_v4_e2"
tr, va, te, adv = (pd.read_csv(f"{D}/{f}.csv") for f in ["train", "val", "test", "test_adversarial"])
k_tr = set(tr.texto_original.map(clave)) | set(tr.texto_modelo.astype(str).map(clave))
for nom, t in [("val", va.texto_original), ("test", te.texto_original), ("batería", adv.texto)]:
    k = set(t.map(clave)) | {clave(normalizar(str(x))) for x in t}
    print(f"solape train ∩ {nom}: {len(k_tr & k)}")
    assert not (k_tr & k)
assert pd.read_csv("data/history/v1.2.0/test.csv").equals(te) and pd.read_csv("data/history/v1.2.0/val.csv").equals(va)
grupo = tr.origen.map(lambda o: "real" if o == "real" else ("octubre" if o == "real_x_oct2026" else "sintético"))
print(pd.crosstab(grupo, tr.clase, margins=True))
assert len(tr) == 10469 and (grupo == "octubre").sum() == 4074, "la composición no coincide con la verificada"

In [ ]:
# 7. Corrida v4: con los datos de octubre (peso 1.0).
#    Guarda los pesos de las 3 semillas (~1.5 GB en Drive) por si se adopta; poné False si no hay espacio.
GUARDAR_MODELOS = True
flag = "--guardar_modelos" if GUARDAR_MODELOS else ""
!python src/train/entrenar_v4.py --peso_nuevos 1.0 --out {RUNS}/v4 {flag} \
    --train data/dataset_v4_e2/train.csv --val data/dataset_v4_e2/val.csv \
    --test data/dataset_v4_e2/test.csv --adv data/dataset_v4_e2/test_adversarial.csv

In [ ]:
# 8. Línea base: mismo archivo, mismas semillas, sin los datos de octubre (peso 0.0)
!python src/train/entrenar_v4.py --peso_nuevos 0.0 --out {RUNS}/base \
    --train data/dataset_v4_e2/train.csv --val data/dataset_v4_e2/val.csv \
    --test data/dataset_v4_e2/test.csv --adv data/dataset_v4_e2/test_adversarial.csv

In [ ]:
# 9. Tabla comparativa, recalculada desde las probabilidades guardadas
!python src/evaluation/comparar_v4.py --base {RUNS}/base --v4 {RUNS}/v4 \
    --test data/dataset_v4_e2/test.csv --adv data/dataset_v4_e2/test_adversarial.csv \
    --salida {RUNS}/comparacion.json

In [ ]:
# 10. Paquete para traer de vuelta: predicciones, probabilidades, métricas y metadata del dataset
!cp data/dataset_v4_e2/metadata.json {RUNS}/metadata_dataset_v4_e2.json
import zipfile, pathlib
with zipfile.ZipFile("/content/resultados_v4.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in pathlib.Path(RUNS).rglob("*"):
        if p.is_file() and "modelo" not in p.parts and "_tmp" not in str(p):
            z.write(p, p.relative_to(RUNS))
print("listo: /content/resultados_v4.zip")
from google.colab import files
files.download("/content/resultados_v4.zip")